In [1]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
real = pd.read_csv("True.csv")
fake = pd.read_csv("Fake.csv")

real["label"] = 1
fake["label"] = 0

data = pd.concat([real, fake], ignore_index=True)

X_train = data["text"].fillna("")
y_train = data["label"]

In [3]:
external = pd.read_csv("external_sources.csv")

external = external[external["language"] == "english"].copy()

X_external = external["text"].fillna("")

In [4]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("lr", LogisticRegression())
])

model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(max_features=5000)),
                ('lr', LogisticRegression())])

In [5]:
external["predicted_label"] = model.predict(X_external)

external["predicted_news"] = external["predicted_label"].map({
    1: "Real",
    0: "Fake"
})

external[["title", "type", "predicted_news"]].head()

,title,type,predicted_news
0,Muslims BUSTED: They Stole Millions In Gov’t B...,bias,Fake
1,Re: Why Did Attorney General Loretta Lynch Ple...,bias,Fake
2,BREAKING: Weiner Cooperating With FBI On Hilla...,bias,Fake
3,PIN DROP SPEECH BY FATHER OF DAUGHTER Kidnappe...,bias,Fake
4,FANTASTIC! TRUMP'S 7 POINT PLAN To Reform Heal...,bias,Fake


In [6]:
# Evaluate External Dataset Predictions
external_results = pd.DataFrame({
    "Prediction": ["Real", "Fake"],
    "Articles": [
        (external["predicted_label"] == 1).sum(),
        (external["predicted_label"] == 0).sum()
    ]
})

external_results["Percentage"] = (
    external_results["Articles"] / len(external) * 100
)

external_results

,Prediction,Articles,Percentage
0,Real,751,6.054987
1,Fake,11652,93.945013


In [7]:
# Analyze Predictions by External Dataset Category
type_results = external.groupby("type")["predicted_label"].value_counts().unstack(fill_value=0)

type_results.columns = ["Predicted Fake", "Predicted Real"] if list(type_results.columns) == [0, 1] else type_results.columns

type_results

,Predicted Fake,Predicted Real
type,,
bias,429,14
bs,10266,634
conspiracy,401,29
fake,17,2
hate,239,4
junksci,101,1
satire,146,0
state,53,67
